# Fase 2: ¿cuánto engaña el reparto al azar?

En la fase 1 encontramos que las 4049 células salen de solo 966 láminas, y que repartiendo las
fotos al azar el 89.8% de las células del examen tienen una hermana del mismo cúmulo en el
entrenamiento.

Acá vamos a medir qué tanto importa eso. La idea es entrenar **el mismo modelo dos veces**,
cambiando una sola cosa:

- **Condición A:** las fotos se reparten al azar, que es como se suele hacer.
- **Condición B:** se reparten por lámina, de manera que las células hermanas nunca queden
  en lados opuestos.

Todo lo demás queda idéntico: mismo modelo, mismas épocas, misma semilla, mismos ajustes. Si los
resultados salen distintos, la única explicación posible es la forma de repartir.

Son 5 entrenamientos por condición (uno por pliegue), 10 en total. Calcula alrededor de una hora
con GPU.

## Preparar el entorno

Antes de correr nada: menú *Entorno de ejecución* → *Cambiar tipo de entorno* → GPU (T4).

In [ ]:
import torch
print("GPU disponible:", torch.cuda.is_available())
print("Modelo:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "sin GPU")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from google.colab import userdata
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')

Recuperamos el índice que armamos en la fase 1 y volvemos a bajar el dataset. Como la carpeta de
descarga puede cambiar de nombre entre sesiones, corregimos las rutas guardadas en el CSV para
que apunten a donde están las imágenes ahora.

In [ ]:
import pandas as pd, kagglehub

DEST = "/content/drive/MyDrive/sipakmed"
df = pd.read_csv(f"{DEST}/indice_celulas.csv")

path = kagglehub.dataset_download("prahladmehandiratta/cervical-cancer-largest-dataset-sipakmed")

# Las rutas del CSV vienen de la sesion anterior: les cambiamos la parte de adelante
base_viejo = df["img"].iloc[0].split("/im_")[0]
for col in ["img", "cyt", "nuc"]:
    df[col] = df[col].str.replace(base_viejo, path, regex=False)

print("Filas:", len(df))
print("Todas las imágenes existen:", df["img"].apply(os.path.exists).all())
df.head(3)

## Los dos repartos

El reparto por lámina ya lo teníamos de la fase 1, en la columna `fold`. Le agregamos ahora el
reparto al azar en una columna nueva, `fold_azar`.

Un detalle importante para que la comparación sea justa: **el reparto al azar también se hace en
cinco pliegues**. No sirve comparar cinco entrenamientos contra uno solo, porque parte de la
diferencia vendría de eso y no del reparto.

In [ ]:
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold

SEMILLA = 42

# Condicion A: al azar, sin mirar de que lamina viene cada celula
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
df["fold_azar"] = -1
for k, (_, val) in enumerate(skf.split(df, df["clase"])):
    df.loc[df.index[val], "fold_azar"] = k

# Condicion B: por lamina (se recalcula igual que en la fase 1, por si el CSV no la trae)
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
df["fold_lamina"] = -1
for k, (_, val) in enumerate(sgkf.split(df, df["clase"], groups=df["lamina"])):
    df.loc[df.index[val], "fold_lamina"] = k

# Control: cuanta contaminacion tiene cada reparto
for col in ["fold_azar", "fold_lamina"]:
    sucias = []
    for k in range(5):
        laminas_train = set(df[df[col] != k]["lamina"])
        sucias.append(df[df[col] == k]["lamina"].isin(laminas_train).mean())
    print(f"{col}: {sum(sucias)/5:.1%} de células con hermanas al otro lado")

Esa salida confirma lo que buscamos: en el reparto al azar casi todas las células están
contaminadas, y en el reparto por lámina el número tiene que ser 0%.

## Cargar las imágenes

Acá hay una decisión que cambia mucho el tiempo de entrenamiento. Lo natural sería abrir cada
foto desde el disco cada vez que se necesita, pero eso significa decodificar 3200 archivos BMP en
cada época, y Colab gratis da solo dos núcleos de CPU. La GPU se quedaría esperando al disco.

En vez de eso decodificamos todo **una sola vez** y lo dejamos en memoria ya escalado a 224x224.
Son unos 600 MB, que entran sin problema, y a partir de ahí cada época se vuelve cuestión de
segundos.

Al entrenar aplicamos volteos, giros y pequeños cambios de color, para que el modelo no memorice
detalles sueltos. Al evaluar no se aplica nada de eso.

Una anotación de la fase 1 que vale repetir: al forzar todo a 224x224 se pierde el tamaño real de
la célula, que es uno de los rasgos que distingue las clases. Lo retomamos en la fase 3.

In [ ]:
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2

df = df.reset_index(drop=True)          # necesario: vamos a indexar por posicion
CLASES = sorted(df["clase"].unique())
IDX = {c: i for i, c in enumerate(CLASES)}
print(CLASES)

# Decodificamos todas las imagenes una sola vez y las guardamos en memoria
print("Cargando imágenes...")
CACHE = np.zeros((len(df), 224, 224, 3), dtype=np.uint8)
for i, ruta in enumerate(df["img"]):
    CACHE[i] = np.array(Image.open(ruta).convert("RGB").resize((224, 224), Image.BILINEAR))
    if i % 500 == 0:
        print(f"  {i}/{len(df)}", end="\r")

Y = df["clase"].map(IDX).values
print(f"Listo: {CACHE.nbytes/1e6:.0f} MB en memoria")

MEDIA = [0.485, 0.456, 0.406]   # valores estandar de ImageNet
DESV  = [0.229, 0.224, 0.225]

tf_train = v2.Compose([
    v2.RandomHorizontalFlip(),
    v2.RandomVerticalFlip(),
    v2.RandomRotation(20),
    v2.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.03),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(MEDIA, DESV),
])

tf_eval = v2.Compose([
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(MEDIA, DESV),
])


class Celulas(Dataset):
    """Lee de memoria, no de disco. Recibe posiciones de fila, no una tabla."""
    def __init__(self, posiciones, tf):
        self.pos = np.asarray(posiciones)
        self.tf = tf

    def __len__(self):
        return len(self.pos)

    def __getitem__(self, i):
        p = self.pos[i]
        img = torch.from_numpy(CACHE[p]).permute(2, 0, 1)   # alto-ancho-color -> color-alto-ancho
        return self.tf(img), Y[p]

## El modelo y el entrenamiento

Usamos EfficientNet-B0 ya preentrenado con ImageNet y le cambiamos la última capa para que
tenga cinco salidas, una por clase. Es un modelo liviano, que acá viene bien porque hay que
entrenarlo diez veces. Además el interés no está en el modelo sino en la comparación entre
repartos.

Dos decisiones a propósito:

- **Épocas fijas, sin parada anticipada.** La parada anticipada necesita apartar un tercer grupo
  de datos y mete otra variable en la comparación. Con épocas fijas queda más limpio.
- **La misma semilla en las diez corridas**, así los pesos iniciales y el orden de los datos no
  explican ninguna diferencia.

In [ ]:
import torch, torch.nn as nn, random, time
from torchvision import models

EPOCAS = 12
LOTE = 64
LR = 3e-4
DEV = "cuda" if torch.cuda.is_available() else "cpu"


def fijar_semilla(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def crear_modelo():
    m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, len(CLASES))
    return m.to(DEV)


def entrenar_y_evaluar(tabla, columna_fold, k):
    fijar_semilla(SEMILLA)

    # Posiciones de fila, porque el cache en memoria se indexa por posicion
    pos_train = np.where(tabla[columna_fold].values != k)[0]
    pos_test  = np.where(tabla[columna_fold].values == k)[0]

    dl_train = DataLoader(Celulas(pos_train, tf_train), batch_size=LOTE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
    dl_test  = DataLoader(Celulas(pos_test, tf_eval), batch_size=128, shuffle=False,
                          num_workers=2, pin_memory=True)

    modelo = crear_modelo()
    opt = torch.optim.AdamW(modelo.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCAS)
    criterio = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler("cuda")

    for ep in range(EPOCAS):
        modelo.train()
        perdida = 0.0
        for x, y in dl_train:
            x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda"):
                salida = modelo(x)
                loss = criterio(salida, y)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            perdida += loss.item() * x.size(0)
        sched.step()
        print(f"    época {ep+1}/{EPOCAS}  pérdida {perdida/len(dl_train.dataset):.4f}", end="\r")

    # Predicciones sobre el pliegue que quedo fuera
    modelo.eval()
    preds, reales = [], []
    with torch.no_grad():
        for x, y in dl_test:
            x = x.to(DEV)
            with torch.amp.autocast("cuda"):
                salida = modelo(x)
            preds.append(salida.argmax(1).cpu().numpy())
            reales.append(y.numpy())

    del modelo; torch.cuda.empty_cache()
    return pos_test, np.concatenate(preds), np.concatenate(reales)

## Correr las diez comparaciones

Cada entrenamiento aprende con cuatro pliegues y rinde examen con el quinto. Como se rota, al
final todas las células pasaron por el examen exactamente una vez en cada condición. Esas
predicciones se guardan en dos columnas nuevas.

Esto tarda. Déjalo corriendo y no cierres la pestaña.

In [ ]:
from sklearn.metrics import f1_score, matthews_corrcoef, accuracy_score

df["pred_azar"] = -1
df["pred_lamina"] = -1
filas_result = []

for nombre, col_fold, col_pred in [("AZAR", "fold_azar", "pred_azar"),
                                   ("LÁMINA", "fold_lamina", "pred_lamina")]:
    print(f"\n=== Condición {nombre} ===")
    for k in range(5):
        t0 = time.time()
        pos, pred, real = entrenar_y_evaluar(df, col_fold, k)
        df.iloc[pos, df.columns.get_loc(col_pred)] = pred

        f1 = f1_score(real, pred, average="macro")
        mcc = matthews_corrcoef(real, pred)
        acc = accuracy_score(real, pred)
        filas_result.append({"condicion": nombre, "pliegue": k,
                             "macro_f1": f1, "mcc": mcc, "accuracy": acc})
        print(f"  pliegue {k}: F1 {f1:.4f} | MCC {mcc:.4f} | acc {acc:.4f}"
              f"  ({time.time()-t0:.0f}s)")

res = pd.DataFrame(filas_result)
res

## Resultados

Promedio de los cinco pliegues con su variación. El "más menos" importa tanto como el promedio:
si las dos condiciones se solapan, la diferencia no es real.

In [ ]:
resumen = res.groupby("condicion")[["macro_f1", "mcc", "accuracy"]].agg(["mean", "std"]).round(4)
print(resumen, "\n")

a = res[res.condicion == "AZAR"]["macro_f1"].mean()
b = res[res.condicion == "LÁMINA"]["macro_f1"].mean()
print(f"Macro F1 con reparto al azar:    {a:.4f}")
print(f"Macro F1 con reparto por lámina: {b:.4f}")
print(f"Caída: {a-b:.4f} ({(a-b)/a:.1%} del valor original)")

Ese último número es el resultado central del trabajo: cuánto del acierto que se reporta
normalmente en este dataset viene de haber visto células hermanas y no de haber aprendido a
reconocer el tipo de célula.

Si la caída resulta chica, también es un resultado válido y se reporta tal cual. Significaría que
el modelo generaliza mejor de lo que temíamos.

## ¿La diferencia es real o es casualidad?

Como todas las células pasaron por el examen en las dos condiciones, se pueden comparar una por
una. La prueba de McNemar mira solo los casos donde las dos condiciones no coinciden: cuántas
acertó una y falló la otra. Si esos desacuerdos están repartidos parejo, la diferencia es ruido.

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar

y = df["clase"].map(IDX).values
ok_azar = (df["pred_azar"].values == y)
ok_lam  = (df["pred_lamina"].values == y)

tabla = [[int((ok_azar & ok_lam).sum()),  int((ok_azar & ~ok_lam).sum())],
         [int((~ok_azar & ok_lam).sum()), int((~ok_azar & ~ok_lam).sum())]]

print("Aciertan las dos:", tabla[0][0])
print("Solo acierta AZAR:", tabla[0][1])
print("Solo acierta LÁMINA:", tabla[1][0])
print("Fallan las dos:", tabla[1][1])

r = mcnemar(tabla, exact=False, correction=True)
print(f"\np = {r.pvalue:.2e}")
print("La diferencia es real" if r.pvalue < 0.05 else "No hay diferencia clara")

## ¿Qué clases se confunden?

Las matrices muestran, para cada tipo de célula real, en qué la clasificó el modelo. La diagonal
son los aciertos. Interesa sobre todo la condición por lámina, que es la honesta.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (titulo, col) in zip(axes, [("Reparto al azar", "pred_azar"),
                                    ("Reparto por lámina", "pred_lamina")]):
    cm = confusion_matrix(y, df[col].values, normalize="true")
    im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(CLASES))); ax.set_yticks(range(len(CLASES)))
    ax.set_xticklabels(CLASES, rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(CLASES, fontsize=8)
    ax.set_title(titulo); ax.set_xlabel("predicho"); ax.set_ylabel("real")
    for i in range(len(CLASES)):
        for j in range(len(CLASES)):
            ax.text(j, i, f"{cm[i,j]:.2f}", ha="center", va="center",
                    color="white" if cm[i, j] > 0.5 else "black", fontsize=8)
plt.tight_layout(); plt.show()

print("Detalle por clase, reparto por lámina:")
print(classification_report(y, df["pred_lamina"].values, target_names=CLASES, digits=3))

## Guardar todo

Guardamos dos archivos: las métricas de cada pliegue y las predicciones célula por célula. Estas
últimas son las que permiten comparar contra los modelos de las fases siguientes sin tener que
volver a entrenar nada.

In [ ]:
res.to_csv(f"{DEST}/resultados_fase2.csv", index=False)
df[["stem", "clase", "lamina", "fold_azar", "fold_lamina",
    "pred_azar", "pred_lamina"]].to_csv(f"{DEST}/predicciones_fase2.csv", index=False)

print("Guardado en", DEST)

## Resumen de la fase 2

Lo que queda demostrado con números propios:

- Entrenando el mismo modelo con las mismas condiciones, lo único que cambia es cómo se reparten
  las fotos, y aun así los resultados difieren.
- La caída de macro F1 entre un reparto y otro es la medida de cuánto se infla el resultado
  cuando no se respetan las láminas.
- Todas las comparaciones de aquí en adelante usan el reparto por lámina, y ya tenemos guardado
  el punto de partida contra el que comparar.

**Lo que sigue (fase 3):** sumarle al modelo las medidas de forma de la célula sacadas de los
contornos (tamaño del núcleo, qué proporción ocupa dentro de la célula, qué tan redondo es) y ver
si mejoran el resultado, sobre todo en la clase que más se confunde.